# ML Recommendation Engine - Hybrid 3-Database System
## Part 1: Setup & Connections

In [1]:
from neo4j import GraphDatabase
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
import pandas as pd
import numpy as np
import os

In [2]:
load_dotenv(find_dotenv(), override=True)

# PostgreSQL
engine = create_engine(os.getenv("PG_CONNECTION"))

# Neo4j
neo4j_driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USER"), os.getenv("NEO4J_PASSWORD"))
)

# Qdrant
client = QdrantClient(host="localhost", port=6333)

model = SentenceTransformer("all-MiniLM-L6-v2")

print("All 3 databases connected!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


All 3 databases connected!


## Strategy 1: Ingredient-Based Recommender (PostgreSQL + Neo4j)

In [3]:
def ingredient_based_recommend(product_id, top_k=5):
    """
    Finds products that share the most high-PageRank ingredients
    with the input product.
    Uses PostgreSQL for ingredient lookup + Neo4j for PageRank weights.
    """
    # Step 1 — Get ingredients for input product from PostgreSQL
    product_ingredients = pd.read_sql(f"""
        SELECT i.ingredient_name
        FROM ingredients i
        JOIN product_ingredients pi ON i.ingredient_id = pi.ingredient_id
        WHERE pi.product_id = '{product_id}';
    """, engine)

    if len(product_ingredients) == 0:
        print(f"No ingredients found for {product_id}")
        return None

    ingredient_list = product_ingredients["ingredient_name"].tolist()
    print(f"Input product has {len(ingredient_list)} ingredients")

    # Step 2 — Get PageRank scores from Neo4j
    with neo4j_driver.session() as session:
        result = session.run("""
            UNWIND $ingredients AS name
            MATCH (i:Ingredient {name: name})
            WHERE i.pagerank IS NOT NULL
            RETURN i.name AS ingredient, i.pagerank AS pagerank
        """, ingredients=ingredient_list)
        pagerank_df = pd.DataFrame([dict(r) for r in result])

    if len(pagerank_df) == 0:
        print("No PageRank scores found")
        return None

    print(f"PageRank scores found for {len(pagerank_df)} ingredients")

    # Step 3 — Find similar products via shared ingredients (PostgreSQL)
    ingredient_str = ",".join([f"'{i}'" for i in ingredient_list])
    similar_products = pd.read_sql(f"""
        SELECT 
            p.product_id,
            p.product_name,
            p.brand_name,
            p.price_usd,
            p.primary_category,
            COUNT(DISTINCT i.ingredient_name) AS shared_ingredients
        FROM products p
        JOIN product_ingredients pi ON p.product_id = pi.product_id
        JOIN ingredients i ON pi.ingredient_id = i.ingredient_id
        WHERE i.ingredient_name IN ({ingredient_str})
        AND p.product_id != '{product_id}'
        GROUP BY p.product_id, p.product_name, p.brand_name, 
                 p.price_usd, p.primary_category
        ORDER BY shared_ingredients DESC
        LIMIT 50;
    """, engine)

    # Step 4 — Weight shared ingredients by PageRank
    pagerank_dict = dict(zip(pagerank_df["ingredient"], pagerank_df["pagerank"]))

    def weighted_score(product_id_check):
        shared = pd.read_sql(f"""
            SELECT i.ingredient_name
            FROM ingredients i
            JOIN product_ingredients pi ON i.ingredient_id = pi.ingredient_id
            WHERE pi.product_id = '{product_id_check}'
            AND i.ingredient_name IN ({ingredient_str});
        """, engine)
        return sum(pagerank_dict.get(ing, 0) 
                   for ing in shared["ingredient_name"].tolist())

    similar_products["pagerank_score"] = similar_products["product_id"].apply(
        weighted_score
    )
    similar_products = similar_products.sort_values(
        "pagerank_score", ascending=False
    ).head(top_k)

    return similar_products

# Test
result1 = ingredient_based_recommend("P420652", top_k=5)
print("\n Ingredient-Based Recommendations:")
print("─" * 60)
print(result1[["product_name", "brand_name", "price_usd", 
               "shared_ingredients", "pagerank_score"]].to_string())

Input product has 34 ingredients
PageRank scores found for 34 ingredients

 Ingredient-Based Recommendations:
────────────────────────────────────────────────────────────
                                                   product_name  brand_name  price_usd  shared_ingredients  pagerank_score
0                                   Berries n' Choco Kisses Set     LANEIGE       26.0                  27        0.037686
2                             Midnight to Morning Hydration Set     LANEIGE       21.0                  26        0.037510
1                                                   Besties Set     LANEIGE       35.0                  26        0.037510
3   BTS |  Amorepacific Lip Sleeping Mask Lip & Pop Edition Set     LANEIGE       35.0                  25        0.037407
14                               Power Starters Tightening Trio  StriVectin       89.0                   7        0.029700


## Strategy 2: Review-Based Recommender (Qdrant)

In [5]:
def review_based_recommend(product_id, top_k=5):
    """
    Finds products with semantically similar reviews.
    Uses Qdrant vector search.
    """
    # Get reviews for input product
    product_reviews = pd.read_sql(f"""
        SELECT review_text
        FROM reviews
        WHERE product_id = '{product_id}'
        AND review_text IS NOT NULL
        LIMIT 20;
    """, engine)

    if len(product_reviews) == 0:
        print(f"No reviews found for {product_id}")
        return None

    # Embed and average to get product vector
    embeddings = model.encode(product_reviews["review_text"].tolist())
    product_vector = embeddings.mean(axis=0).tolist()

    # Search Qdrant excluding source product
    results = client.query_points(
        collection_name="sephora_reviews",
        query=product_vector,
        query_filter=Filter(
            must_not=[
                FieldCondition(
                    key="product_id",
                    match=MatchValue(value=product_id)
                )
            ]
        ),
        limit=100
    ).points

    # Get unique products with avg score
    seen = {}
    for r in results:
        pid = r.payload["product_id"]
        if pid not in seen:
            seen[pid] = []
        seen[pid].append(r.score)

    # Average scores per product
    product_scores = {
        pid: np.mean(scores) 
        for pid, scores in seen.items()
    }
    top_products = sorted(
        product_scores.items(), key=lambda x: x[1], reverse=True
    )[:top_k]

    if not top_products:
        return None

    # Look up product details
    placeholders = ",".join([f"'{p[0]}'" for p in top_products])
    product_details = pd.read_sql(f"""
        SELECT product_id, product_name, brand_name, price_usd
        FROM products
        WHERE product_id IN ({placeholders});
    """, engine)

    score_df = pd.DataFrame(top_products, columns=["product_id", "semantic_score"])
    result = product_details.merge(score_df, on="product_id")
    result = result.sort_values("semantic_score", ascending=False)

    return result

# Test
result2 = review_based_recommend("P420652", top_k=5)
print("\n Review-Based Recommendations:")
print("─" * 60)
print(result2[["product_name", "brand_name", 
               "price_usd", "semantic_score"]].to_string())


 Review-Based Recommendations:
────────────────────────────────────────────────────────────
                      product_name           brand_name  price_usd  semantic_score
1                   Lip Glowy Balm              LANEIGE       18.0        0.845242
2                    Rosebud Salve  Rosebud Perfume Co.        7.0        0.842955
0  Intense Therapy Lip Balm SPF 25           Jack Black       10.0        0.840946


## Strategy 3: Hybrid Recommender

In [6]:
def hybrid_recommend(product_id, skin_type=None, top_k=5):
    ing_recs = ingredient_based_recommend(product_id, top_k=20)
    rev_recs = review_based_recommend(product_id, top_k=20)

    if ing_recs is None or rev_recs is None:
        print("Could not generate recommendations")
        return

    # Normalize scores
    ing_recs["ing_score_norm"] = (
        ing_recs["pagerank_score"] / ing_recs["pagerank_score"].max()
    )
    rev_recs["sem_score_norm"] = (
        rev_recs["semantic_score"] / rev_recs["semantic_score"].max()
    )

    # Merge
    hybrid = ing_recs[["product_id", "product_name", "brand_name",
                        "price_usd", "ing_score_norm"]].merge(
        rev_recs[["product_id", "sem_score_norm"]],
        on="product_id",
        how="outer"
    ).fillna(0)

    # Fix missing product details from Qdrant-only results
    missing = hybrid[hybrid["product_name"] == 0]["product_id"].tolist()
    if missing:
        placeholders = ",".join([f"'{p}'" for p in missing])
        missing_details = pd.read_sql(f"""
            SELECT product_id, product_name, brand_name, price_usd
            FROM products
            WHERE product_id IN ({placeholders});
        """, engine)
        for _, row in missing_details.iterrows():
            mask = hybrid["product_id"] == row["product_id"]
            hybrid.loc[mask, "product_name"] = row["product_name"]
            hybrid.loc[mask, "brand_name"] = row["brand_name"]
            hybrid.loc[mask, "price_usd"] = row["price_usd"]

    # Compute hybrid score
    hybrid["hybrid_score"] = (
        0.5 * hybrid["ing_score_norm"] +
        0.5 * hybrid["sem_score_norm"]
    )

    # Skin type filter
    if skin_type:
        skin_products = pd.read_sql(f"""
            SELECT DISTINCT product_id
            FROM product_skin_types
            WHERE LOWER(skin_type) = LOWER('{skin_type}');
        """, engine)
        skin_ids = skin_products["product_id"].tolist()
        hybrid = hybrid[hybrid["product_id"].isin(skin_ids)]

    hybrid = hybrid.sort_values("hybrid_score", ascending=False).head(top_k)

    print(f"\nTop {top_k} Hybrid Recommendations:")
    print("─" * 60)
    print(hybrid[["product_name", "brand_name", 
                  "price_usd", "hybrid_score"]].to_string())

    return hybrid

## Full Pipeline Demo

In [7]:
def full_recommendation_pipeline(product_id, skin_type):
    """
    Complete pipeline using all 3 databases.
    Given a product + skin type → recommend similar products.
    """
    # Get input product name
    product_info = pd.read_sql(f"""
        SELECT product_name, brand_name, price_usd, primary_category
        FROM products
        WHERE product_id = '{product_id}';
    """, engine)

    print("=" * 60)
    print("SEPHORA MULTI-DATABASE RECOMMENDATION ENGINE")
    print("=" * 60)
    print(f"Input Product : {product_info.iloc[0]['product_name']}")
    print(f"Brand         : {product_info.iloc[0]['brand_name']}")
    print(f"Price         : ${product_info.iloc[0]['price_usd']}")
    print(f"Category      : {product_info.iloc[0]['primary_category']}")
    print(f"Skin Type     : {skin_type}")
    print("=" * 60)

    print("\n Step 1: PostgreSQL — fetching ingredients...")
    print(" Step 2: Neo4j — applying PageRank weights...")
    print(" Step 3: Qdrant — semantic review matching...")
    print(" Step 4: Hybrid scoring — combining signals...")
    print()

    recommendations = hybrid_recommend(product_id, skin_type=skin_type, top_k=5)

    print("\n" + "=" * 60)
    print("FINAL RECOMMENDATIONS")
    print("=" * 60)

    if recommendations is not None:
        for idx, row in recommendations.iterrows():
            print(f"\n#{list(recommendations.index).index(idx)+1}")
            print(f"  Product : {row['product_name']}")
            print(f"  Brand   : {row['brand_name']}")
            print(f"  Price   : ${row['price_usd']}")
            print(f"  Score   : {row['hybrid_score']:.3f}")

# Run the full pipeline
full_recommendation_pipeline("P420652", skin_type="dry")

SEPHORA MULTI-DATABASE RECOMMENDATION ENGINE
Input Product : Lip Sleeping Mask Intense Hydration with Vitamin C
Brand         : LANEIGE
Price         : $24.0
Category      : Skincare
Skin Type     : dry

 Step 1: PostgreSQL — fetching ingredients...
 Step 2: Neo4j — applying PageRank weights...
 Step 3: Qdrant — semantic review matching...
 Step 4: Hybrid scoring — combining signals...

Input product has 34 ingredients
PageRank scores found for 34 ingredients

Top 5 Hybrid Recommendations:
────────────────────────────────────────────────────────────
                         product_name           brand_name  price_usd  hybrid_score
21        Berries n' Choco Kisses Set              LANEIGE       26.0      0.500000
4                      Lip Glowy Balm              LANEIGE       18.0      0.500000
1                       Rosebud Salve  Rosebud Perfume Co.        7.0      0.498647
20  Midnight to Morning Hydration Set              LANEIGE       21.0      0.497664
0     Intense Therapy Li